In [ ]:
#膵腫瘍画像診断AIの構築
from google.colab import drive
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models, utils
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
import matplotlib.pyplot as plt
import numpy as np


# 1 Google Driveをマウント（まだの場合）
drive.flush_and_unmount()
drive.mount('/content/drive')

# --- 設定 ---
BATCH_SIZE = 32
SAVE_PATH = '/content/drive/MyDrive/テックアカデミー/model_checkpoints' # 保存先
LOG_DIR = '/content/drive/MyDrive/テックアカデミー/logs' # TensorBoardログ先
# os.makedirs(SAVE_PATH, exist_ok=True)

# 2. TensorBoardの準備
writer = SummaryWriter(LOG_DIR)

# 3. GPUの設定
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用デバイス: {device}")

# 4. データの前処理（3つのモデル共通で224x224を使います）
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(15),      # データ拡張：少し回転
    transforms.RandomHorizontalFlip(),  # データ拡張：左右反転
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 5. データセットの読み込み
# パスはご自身の環境に合わせて変更してください
train_dir = '/content/drive/MyDrive/テックアカデミー/dataset/pancreas_ultrasound_image/train'
val_dir = '/content/drive/MyDrive/テックアカデミー/dataset/pancreas_ultrasound_image/validation'
train_dataset = datasets.ImageFolder(train_dir, transform=transform_train)
val_dataset = datasets.ImageFolder(val_dir, transform=transform_val)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)


# 逆正規化用の関数（可視化用）
def imshow_tensorboard(img_tensor, title):
    img = img_tensor.clone().cpu()
    # 標準化を戻す (mean, std)
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img = img * std + mean
    return img

# 6. 拡張された学習用関数
def train_model(model, criterion, optimizer, num_epochs=30):
    model = model.to(device)
    best_acc = 0.0  # ベスト精度を記録用

    for epoch in range(num_epochs):
        # --- Training Phase ---
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device).float().unsqueeze(1)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            # シグモイド後の値が0.5以上ならクラス1
            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct_train += torch.sum(preds == labels.data)
            total_train += labels.size(0)

        epoch_loss = running_loss / len(train_dataset)
        epoch_acc = correct_train.double() / total_train

        # --- Validation Phase ---
        model.eval()
        val_loss = 0.0
        correct_val = 0
        total_val = 0

        # 可視化用に1バッチ分だけデータを保持
        images_to_show = None
        labels_to_show = None
        preds_to_show = None

        with torch.no_grad():
            for i, (inputs, labels) in enumerate(val_loader):
                inputs, labels = inputs.to(device), labels.to(device).float().unsqueeze(1)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)
                preds = (torch.sigmoid(outputs) > 0.5).float()
                correct_val += torch.sum(preds == labels.data)
                total_val += labels.size(0)

                # 最初のバッチを可視化用に保存
                if i == 0:
                    images_to_show = inputs
                    labels_to_show = labels
                    preds_to_show = preds

        val_epoch_loss = val_loss / len(val_dataset)
        val_epoch_acc = correct_val.double() / total_val

        # --- TensorBoardへの記録 ---
        writer.add_scalars('Loss', {'train': epoch_loss, 'val': val_epoch_loss}, epoch)
        writer.add_scalars('Accuracy', {'train': epoch_acc, 'val': val_epoch_acc}, epoch)

        # 推論結果の画像をTensorBoardへ送る (エポックごと)
        grid = utils.make_grid([imshow_tensorboard(img, "") for img in images_to_show[:8]]) # 最初の8枚
        writer.add_image(f'Val_Inference_Epoch_{epoch}', grid, epoch)
        # テキストで予測結果を記録 (例: "Pred: 1, Label: 0")
        sample_results = f"Epoch {epoch}: Preds: {preds_to_show[:8].flatten().tolist()} / Labels: {labels_to_show[:8].flatten().tolist()}"
        writer.add_text('Inference_Sample', sample_results, epoch)

        print(f'Epoch {epoch+1}/{num_epochs} | Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f} | Val Loss: {val_epoch_loss:.4f} Acc: {val_epoch_acc:.4f}')

        # --- ベストモデルの保存 ---
        if val_epoch_acc > best_acc:
            best_acc = val_epoch_acc
            model_name = model.__class__.__name__
            torch.save(model.state_dict(), os.path.join(SAVE_PATH, f'best_{model_name}.pth'))
            print(f"--> Best model saved with Acc: {best_acc:.4f}")

    writer.close()
    print("学習終了！")

In [ ]:
# --- ResNet-18モデルの構築 ---
print("ResNet-18の学習を開始します...")

# 1. 学習済みモデルをロード
model_resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# 2. 全層を凍結
for param in model_resnet.parameters():
    param.requires_grad = False

# 3. 最後の全結合層（fc）を書き換え
# ResNet18の最終層入力は512次元
num_ftrs = model_resnet.fc.in_features
model_resnet.fc = nn.Sequential(
    nn.Linear(num_ftrs, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, 1)
)

# 4. 設定と実行
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model_resnet.fc.parameters(), lr=0.001)

train_model(model_resnet, criterion, optimizer, num_epochs=30)

In [ ]:
# --- EfficientNet-V2 (Small) モデルの構築 ---
print("EfficientNet-V2の学習を開始します...")

# 1. 学習済みモデルをロード
model_eff = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)

# 2. 全層を凍結
for param in model_eff.parameters():
    param.requires_grad = False

# 3. 分類器（classifier）を書き換え
# EfficientNetのclassifierは [Dropout, Linear] の構造です
num_ftrs = model_eff.classifier[1].in_features
model_eff.classifier[1] = nn.Sequential(
    nn.Linear(num_ftrs, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, 1)
)

# 4. 設定と実行
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model_eff.classifier[1].parameters(), lr=0.001)

train_model(model_eff, criterion, optimizer, num_epochs=30)

Tensorboardを表示

In [ ]:
# TensorBoardの拡張をロード
%load_ext tensorboard

# ログディレクトリを指定して起動
%tensorboard --logdir '/content/drive/MyDrive/テックアカデミー/logs'

検証

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import os

# ==========================================
# 1. 推論用クラスの定義
# ==========================================
class AIInferenceEngine:
    def __init__(self, model, weights_path, class_names, device=None):
        self.device = device if device else torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)

        # 重みのロード
        self.model.load_state_dict(torch.load(weights_path, map_location=self.device))
        self.model.eval()

        self.class_names = class_names

        # 前処理を学習時と統一
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

    def predict(self, image_path):
        img = Image.open(image_path).convert('RGB')
        img_tensor = self.transform(img).unsqueeze(0).to(self.device)

        with torch.no_grad():
            outputs = self.model(img_tensor)
            # 出力が1つの場合はSigmoidで0~1の確率に変換
            probability = torch.sigmoid(outputs).squeeze().item()

        # 0.5をしきい値として判定
        if probability >= 0.5:
            idx = 1
            conf = probability
        else:
            idx = 0
            conf = 1 - probability

        return img, self.class_names[idx], conf

# ==========================================
# 2. 実行設定
# ==========================================
target_classes = ['normal', 'tumor']

# モデルのインスタンス化 (ここでは例としてダミーの構造を想定)
# ※実際には学習時に使用したモデルクラスをここで定義・インポートしてください
from torchvision import models

model_resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# 2. 全層を凍結
for param in model_resnet.parameters():
    param.requires_grad = False

# 3. 最後の全結合層（fc）を書き換え
# ResNet18の最終層入力は512次元
num_ftrs_res = model_resnet.fc.in_features
model_resnet.fc = nn.Sequential(
    nn.Linear(num_ftrs_res, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, 1)
)

model_eff = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)

# 2. 全層を凍結
for param in model_eff.parameters():
    param.requires_grad = False

# 3. 分類器（classifier）を書き換え
# EfficientNetのclassifierは [Dropout, Linear] の構造です
num_ftrs_eff = model_eff.classifier[1].in_features
model_eff.classifier[1] = nn.Sequential(
    nn.Linear(num_ftrs_eff, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, 1)
)



# モデルと重みのマッピング
models_to_test = {
    "ResNet18": {"model": model_resnet, "weights": "/content/drive/MyDrive/テックアカデミー/model_checkpoints/best_ResNet.pth"},
    "EfficientNet": {"model": model_eff, "weights": "/content/drive/MyDrive/テックアカデミー/model_checkpoints/best_EfficientNet.pth"}
}

# 推論したい画像のパス
test_image_path = "/content/drive/MyDrive/テックアカデミー/dataset/pancreas_ultrasound_image/validation/tumor/PK88.png"

# ==========================================
# 3. 推論の実行と可視化
# ==========================================
def run_comparison(image_path, model_dict):
    num_models = len(model_dict)
    fig, axes = plt.subplots(1, num_models, figsize=(6 * num_models, 5))
    if num_models == 1: axes = [axes]

    for i, (name, config) in enumerate(model_dict.items()):
        try:
            # エンジンの初期化と推論
            engine = AIInferenceEngine(config['model'], config['weights'], target_classes)
            raw_img, label, confidence = engine.predict(image_path)

            # 結果の表示
            axes[i].imshow(raw_img)
            axes[i].set_title(f"Model: {name}\nPredict: {label}\nConf: {confidence:.2%}",
                             fontsize=12, fontweight='bold', color='darkblue')
            axes[i].axis('off')
        except Exception as e:
            print(f"Error processing {name}: {e}")

    plt.tight_layout()
    plt.show()

# 実行
if os.path.exists(test_image_path):
    run_comparison(test_image_path, models_to_test)
else:
    print(f"画像ファイルが見つかりません: {test_image_path}")

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd

def plot_confusion_matrix(model, dataloader, class_names, model_name, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            # Sigmoidで確率に変換し、0.5以上をクラス1とする
            probs = torch.sigmoid(outputs).squeeze()
            preds = (probs >= 0.5).int()

            # テンソルをCPUに移動してリストに追加
            # (バッチサイズが1の場合の対策としてcpu().numpy().flatten()を使用)
            all_preds.extend(preds.cpu().numpy().flatten().tolist())
            all_labels.extend(labels.numpy().flatten().tolist())

    # 混同行列の計算
    cm = confusion_matrix(all_labels, all_preds)

    # 可視化
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
    plt.ylabel('True Label', fontsize=12, fontweight='bold')
    plt.title(f'Confusion Matrix: {model_name}', fontsize=14)
    plt.show()

    # 詳細なレポート（適合率、再現率、F1スコアなど）の表示
    print(f"\n--- Classification Report: {model_name} ---")
    print(classification_report(all_labels, all_preds, target_names=class_names))

for name, config in models_to_test.items():
    print(f"Testing {name}...")

    # 重みのロード
    model = config['model']
    model.load_state_dict(torch.load(config['weights'], map_location=device))
    model.to(device)

    # 混同行列の表示
    plot_confusion_matrix(model, val_loader, target_classes, name, device)